# Interactive ray trajectories for imperfect mirrors

This notebook traces blue (450 nm), green (530 nm), and red (650 nm) rays from air onto four two-dimensional mirror profiles. Each mirror has its own interactive section.

Use **incident angle** to rotate the incoming collimated beam. The angle is measured from the downward-facing normal at the mirror vertex: positive values tilt the beam toward +x. Use **surface perfectness** from 0 to 1, where 1 is the ideal analytic profile and smaller values add a fixed, repeatable multi-scale height error.

The plotted paths use the vector law of reflection, $\mathbf{d}_r=\mathbf{d}_i-2(\mathbf{d}_i\cdot\hat{\mathbf{n}})\hat{\mathbf{n}}$. Optiland is used, when installed, to maintain matching ideal conic reference models for later optical-design analysis; the explicit NumPy tracer makes the imperfect surface model transparent and responsive.

Run the notebook from top to bottom, then move the sliders in any mirror section.

## 0. Installation

Run the next cell once if these packages are not already available, then restart the kernel if Jupyter asks you to. Optiland is optional for the interactive plot but enables the matching reference-model objects.

In [ ]:
%pip install -q numpy matplotlib ipywidgets "optiland>=0.6,<0.7"

In [1]:
from importlib.metadata import PackageNotFoundError, version

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

try:
    from optiland import optic as optiland_optic
    OPTILAND_AVAILABLE = True
    OPTILAND_VERSION = version("optiland")
except (ImportError, PackageNotFoundError):
    optiland_optic = None
    OPTILAND_AVAILABLE = False
    OPTILAND_VERSION = "not installed"

plt.rcParams.update({
    "figure.figsize": (10.5, 6.2),
    "axes.grid": True,
    "grid.alpha": 0.20,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Optiland: {OPTILAND_VERSION}")

Optiland: 0.6.2


## 1. Shared optical model

All dimensions are in millimetres. The mirrors share a 9 mm clear aperture so their behaviour is easy to compare. For visibility, the three wavelengths strike three neighbouring zones of the mirror; ideal metallic-mirror geometry is wavelength independent, so coincident rays would otherwise overlap exactly.

The imperfection is a deterministic sum of long-, medium-, and short-spatial-frequency sinusoidal figure errors. At perfectness 0 its maximum possible envelope is 0.22 mm (deliberately exaggerated for teaching); at perfectness 1 it is exactly zero. Both the perturbed height and its local slope affect reflection.

In [2]:
APERTURE = 4.5  # half-width [mm]
MAX_ERROR_MM = 0.22

SHAPES = {
    "flat": {"label": "Flat"},
    "circular": {"label": "Circular", "radius": 8.0},
    "parabolic": {"label": "Parabolic", "focal_length": 4.0},
    "ellipsoidal": {"label": "Ellipsoidal", "a": 6.5, "b": 8.0},
}

SPECTRAL_RAYS = (
    ("Blue", 450, "#1565ff", -0.56),
    ("Green", 530, "#00a651", 0.00),
    ("Red", 650, "#e32636", +0.56),
)

def ideal_profile_and_slope(shape, x):
    """Return ideal mirror height y(x) and dy/dx."""
    x = np.asarray(x, dtype=float)

    if shape == "flat":
        return np.zeros_like(x), np.zeros_like(x)

    if shape == "circular":
        radius = SHAPES[shape]["radius"]
        root = np.sqrt(np.maximum(radius**2 - x**2, 1e-12))
        return radius - root, x / root

    if shape == "parabolic":
        focal_length = SHAPES[shape]["focal_length"]
        return x**2 / (4.0 * focal_length), x / (2.0 * focal_length)

    if shape == "ellipsoidal":
        a = SHAPES[shape]["a"]       # transverse semi-axis
        b = SHAPES[shape]["b"]       # axial semi-axis
        root = np.sqrt(np.maximum(1.0 - (x / a)**2, 1e-12))
        height = b * (1.0 - root)
        slope = b * x / (a**2 * root)
        return height, slope

    raise ValueError(f"Unknown mirror shape: {shape}")

def figure_error_and_slope(x, perfectness):
    """Repeatable surface-height error and its derivative."""
    x = np.asarray(x, dtype=float)
    u = x / APERTURE
    amplitude = MAX_ERROR_MM * (1.0 - float(perfectness))
    frequencies = np.array([1.35, 3.10, 7.25])
    weights = np.array([0.55, 0.30, 0.15])
    phases = np.array([0.25, -0.70, 1.10])
    arguments = 2.0 * np.pi * frequencies[:, None] * np.atleast_1d(u) + phases[:, None]
    error = amplitude * np.sum(weights[:, None] * np.sin(arguments), axis=0)
    slope = amplitude / APERTURE * np.sum(
        weights[:, None] * 2.0 * np.pi * frequencies[:, None] * np.cos(arguments),
        axis=0,
    )
    if x.ndim == 0:
        return float(error[0]), float(slope[0])
    return error, slope

def actual_profile_and_slope(shape, x, perfectness):
    ideal_height, ideal_slope = ideal_profile_and_slope(shape, x)
    error_height, error_slope = figure_error_and_slope(x, perfectness)
    return ideal_height + error_height, ideal_slope + error_slope

def conic_parameters(shape):
    """Radius and conic constant for the matching Optiland ideal surface."""
    if shape == "flat":
        return np.inf, 0.0
    if shape == "circular":
        return SHAPES[shape]["radius"], 0.0
    if shape == "parabolic":
        return 2.0 * SHAPES[shape]["focal_length"], -1.0
    if shape == "ellipsoidal":
        a, b = SHAPES[shape]["a"], SHAPES[shape]["b"]
        return a**2 / b, -(1.0 - a**2 / b**2)
    raise ValueError(shape)

OPTILAND_REFERENCES = {}

def build_optiland_reference(shape, incident_angle_deg):
    """Build an ideal sequential Optiland conic matching the displayed mirror."""
    if not OPTILAND_AVAILABLE:
        return None

    radius, conic = conic_parameters(shape)
    model = optiland_optic.Optic(name=f"{SHAPES[shape]['label']} mirror reference")
    model.set_aperture(aperture_type="EPD", value=2.0 * APERTURE)
    model.set_field_type(field_type="angle")
    model.add_field(y=float(incident_angle_deg))
    for wavelength_um in (0.450, 0.530, 0.650):
        model.add_wavelength(value=wavelength_um, is_primary=(wavelength_um == 0.530))
    model.add_surface(index=0, radius=np.inf, thickness=np.inf)
    model.add_surface(
        index=1, radius=radius, conic=conic, thickness=-12.0,
        material="mirror", is_stop=True,
    )
    model.add_surface(index=2)
    return model

In [3]:
def reflected_direction(incoming, slope):
    """Specular reflection from a surface whose tangent is (1, slope)."""
    normal = np.array([-slope, 1.0], dtype=float)
    normal /= np.linalg.norm(normal)
    outgoing = incoming - 2.0 * np.dot(incoming, normal) * normal
    return outgoing / np.linalg.norm(outgoing)

def trace_and_plot(shape, incident_angle_deg=0.0, perfectness=1.0):
    """Trace three visible-wavelength rays and draw the selected mirror."""
    if shape not in SHAPES:
        raise ValueError(f"shape must be one of {tuple(SHAPES)}")
    perfectness = float(np.clip(perfectness, 0.0, 1.0))
    angle = np.deg2rad(float(incident_angle_deg))
    incoming = np.array([np.sin(angle), -np.cos(angle)], dtype=float)

    surface_x = np.linspace(-APERTURE, APERTURE, 1201)
    ideal_y, _ = ideal_profile_and_slope(shape, surface_x)
    surface_y, _ = actual_profile_and_slope(shape, surface_x, perfectness)
    rms_error_um = 1000.0 * np.sqrt(np.mean((surface_y - ideal_y)**2))

    optiland_note = "Optiland reference: unavailable (NumPy tracer remains active)"
    if OPTILAND_AVAILABLE:
        try:
            OPTILAND_REFERENCES[shape] = build_optiland_reference(shape, incident_angle_deg)
            optiland_note = f"Optiland {OPTILAND_VERSION} ideal-conic reference: updated"
        except Exception as exc:
            optiland_note = f"Optiland reference not built: {type(exc).__name__}"

    fig, ax = plt.subplots(figsize=(10.5, 6.2))
    if perfectness < 0.999:
        ax.plot(surface_x, ideal_y, color="0.55", lw=1.5, ls="--", label="Ideal profile")
    ax.plot(surface_x, surface_y, color="0.12", lw=5.0, solid_capstyle="round", label="Mirror surface")

    all_points = [np.column_stack((surface_x, surface_y))]
    for color_name, wavelength_nm, color, impact_fraction in SPECTRAL_RAYS:
        hit_x = impact_fraction * APERTURE
        hit_y, slope = actual_profile_and_slope(shape, hit_x, perfectness)
        hit = np.array([hit_x, hit_y], dtype=float)
        origin = hit - 6.5 * incoming
        outgoing = reflected_direction(incoming, slope)
        endpoint = hit + 8.5 * outgoing

        ax.plot(
            [origin[0], hit[0]], [origin[1], hit[1]],
            color=color, lw=2.3, label=f"{color_name} ({wavelength_nm} nm)",
        )
        ax.plot([hit[0], endpoint[0]], [hit[1], endpoint[1]], color=color, lw=2.3)
        ax.scatter(hit[0], hit[1], s=30, color=color, edgecolor="white", linewidth=0.7, zorder=5)
        all_points.append(np.vstack((origin, hit, endpoint)))

    points = np.vstack(all_points)
    x_span = max(np.ptp(points[:, 0]), 1.0)
    y_span = max(np.ptp(points[:, 1]), 1.0)
    ax.set_xlim(points[:, 0].min() - 0.08 * x_span, points[:, 0].max() + 0.08 * x_span)
    ax.set_ylim(points[:, 1].min() - 0.08 * y_span, points[:, 1].max() + 0.12 * y_span)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    ax.set_title(
        f"{SHAPES[shape]['label']} mirror | incidence {incident_angle_deg:+.1f} degrees | "
        f"perfectness {perfectness:.2f}"
    )
    ax.text(
        0.01, 0.99,
        f"Model RMS surface error: {rms_error_um:.1f} um\n{optiland_note}",
        transform=ax.transAxes, va="top", ha="left", fontsize=9,
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "alpha": 0.82, "edgecolor": "0.8"},
    )
    ax.legend(loc="best", framealpha=0.92)
    plt.show()

def show_mirror_controls(shape, default_angle=0.0, default_perfectness=1.0):
    """Display independent controls and an output plot for one mirror section."""
    angle_slider = widgets.FloatSlider(
        value=default_angle, min=-35.0, max=35.0, step=1.0,
        description="Incident angle", readout_format=".1f",
        continuous_update=False, style={"description_width": "initial"},
        layout=widgets.Layout(width="390px"),
    )
    perfectness_slider = widgets.FloatSlider(
        value=default_perfectness, min=0.0, max=1.0, step=0.02,
        description="Surface perfectness", readout_format=".2f",
        continuous_update=False, style={"description_width": "initial"},
        layout=widgets.Layout(width="390px"),
    )

    def update(incident_angle, surface_perfectness):
        trace_and_plot(shape, incident_angle, surface_perfectness)

    output = widgets.interactive_output(
        update,
        {"incident_angle": angle_slider, "surface_perfectness": perfectness_slider},
    )
    controls = widgets.HBox(
        [angle_slider, perfectness_slider],
        layout=widgets.Layout(display="flex", flex_flow="row wrap", gap="10px"),
    )
    display(widgets.VBox([controls, output]))

## 2. Flat mirror

The ideal surface is $y=0$. At perfectness 1, all three incoming parallel rays remain parallel after reflection. Reducing perfectness changes the local normal at each impact point, producing angular spread.

In [10]:
show_mirror_controls("flat", default_angle=0.0, default_perfectness=1.0)

## 3. Circular mirror

This is a concave circular arc with radius 8 mm: $y=R-\sqrt{R^2-x^2}$. Near the axis, normally incident rays approach the paraxial focus at $R/2$; marginal rays reveal spherical aberration.

In [5]:
show_mirror_controls("circular", default_angle=0.0, default_perfectness=1.0)

## 4. Parabolic mirror

The ideal profile is $y=x^2/(4f)$ with focal length $f=4$ mm. At zero incidence and perfectness 1, every displayed parallel ray reflects through the geometric focus $(0,f)$. Tilt introduces off-axis aberration; figure error perturbs the convergence.

In [ ]:
show_mirror_controls("parabolic", default_angle=0.0, default_perfectness=1.0)

## 5. Ellipsoidal mirror

The meridional profile is the lower vertex of an ellipse with transverse semi-axis $a=6.5$ mm and axial semi-axis $b=8$ mm: $y=b[1-\sqrt{1-(x/a)^2}]$. An ideal full ellipsoid maps rays from one focus to the other; this collimated-beam view instead highlights how its curvature differs from the circular and parabolic cases.

In [7]:
show_mirror_controls("ellipsoidal", default_angle=0.0, default_perfectness=1.0)

## 6. Physics checks and interpretation

The next cell verifies two useful limiting cases: a normal ray reverses direction at a perfect flat mirror, and normally incident rays reflect from a perfect parabola toward its focus.

This is a 2-D geometric-optics model. The colors identify trajectories and wavelengths, but an ideal mirror's geometric reflection direction is not dispersive. The model does not include wavelength-dependent coating reflectivity, diffraction, polarization, or scattering from micro-roughness. The perfectness slider represents macroscopic figure/slope error and is intentionally exaggerated; replace `MAX_ERROR_MM` or `figure_error_and_slope` with measured surface data for an engineering tolerance study.

In [8]:
incoming_normal = np.array([0.0, -1.0])
assert np.allclose(reflected_direction(incoming_normal, slope=0.0), [0.0, 1.0])

focus = np.array([0.0, SHAPES["parabolic"]["focal_length"]])
for test_x in (-3.0, -1.5, 0.0, 1.5, 3.0):
    test_y, test_slope = ideal_profile_and_slope("parabolic", test_x)
    hit = np.array([test_x, test_y])
    outgoing = reflected_direction(incoming_normal, test_slope)
    direction_to_focus = focus - hit
    direction_to_focus /= np.linalg.norm(direction_to_focus)
    assert np.allclose(outgoing, direction_to_focus, atol=1e-12)

print("Physics checks passed: flat reflection and parabolic focusing.")

Physics checks passed: flat reflection and parabolic focusing.
